# Generator Logs: Debug Statistics Analysis (04-10-2025)

This notebook analyzes JSON logs from a **specific snapshot** of generator performance data, captured on **October 4, 2025**.

**Analysis versioning structure:**
- Log files are stored in dated folders (`logs/2025-10-12/`) to preserve historical snapshots
- Each analysis notebook is tied to a specific log folder via filename suffix (`debug_statistics_analysis__2025-10-12.ipynb`)
- When generator refinements are implemented, new logs are collected in a new dated folder with a corresponding notebook copy
- This approach ensures reproducibility: re-running this notebook will always analyze the same data that informed the documented observations and recommendations

**Contents:**
- Aggregated performance metrics across configuration parameters
- Quality evaluation and ranking using multi-factor scoring
- Manual analysis of generated repository names with identified issues
- Recommendations for generator improvements

**How to use:**
1. Ensure the `logs/2025-10-12/` folder contains the JSON log files for this analysis
2. Run the cells sequentially to reproduce the analysis
3. Note: If generator improvements have been implemented, those changes will NOT be reflected in this analysis—refer to newer dated notebooks for updated evaluations

**Note:**

The default rendering of markdown text size in vs-code is quite large, to change it into more appropriate font follow the instructions below:
1) Open Settings (`Ctrl+,`).
2) Search: **Notebook Markup: Font Size**.
3) Set a value (e.g., `15`).
4) (If needed) Reload window: `Ctrl+Shift+P` → “Developer: Reload Window”.


***
# <span style="color: #f91974  ; font-weight: bold; font-size: 32px;">0.</span> <span style="color:rgb(186, 176, 115)  ; font-size: 32px;"><em>Initialization</em></span> 

This section establishes the data analysis environment and loads generator performance logs for evaluation.

**Setup tasks:**
- Import required libraries
- Configure pandas display settings for readability
- Load all debug logs from the `logs/2025-10-12/` directory into a structured DataFrame
- Parse JSON log files and extract configuration parameters, performance metrics, and generated samples
- Compute derived metrics (ratios, percentages) for comparative analysis

**Data structure:**
Each row represents a single generator run with its configuration (k-value, temperature, EOS settings), performance metrics (F1 score, Levenshtein distance, generation time), and sample outputs.

In [4]:
# IMPORTS
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

# SETTINGS
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

In [5]:
# helper function
def analyze_delimiter_issues(samples):
    """Check for delimiter-related quality issues in samples."""
    if not samples:
        return {'ends_with_delimiter_ratio': 0, 'short_segment_ratio': 0}
    
    ends_with_delimiter = 0
    short_segment = 0
    
    import re
    for s in samples:
        if not s:
            continue
        # Check if ends with delimiter
        if s.endswith('-') or s.endswith('_'):
            ends_with_delimiter += 1
        
        # Check for 1-2 char segments after delimiters
        segments = re.split(r'[-_]', s)
        for seg in segments[1:]:  # Skip first segment (the seed)
            if len(seg) > 0 and len(seg) <= 2:
                short_segment += 1
                break  # Count once per sample
    
    return {
        'ends_with_delimiter_ratio': ends_with_delimiter / len(samples),
        'short_segment_ratio': short_segment / len(samples)
    }

In [6]:
# LOAD LOGS INTO DATAFRAME
LOG_DIR = Path('logs/2025-10-12')
files = sorted(LOG_DIR.glob('generator_debug_*.json'))
print(f"Found {len(files)} log file(s)")

rows = []
for p in files:
    try:
        with p.open('r', encoding='utf-8') as f:
            data = json.load(f)
        ts = data.get('timestamp')
        k = data.get('k')
        seed = data.get('seed')
        max_len = data.get('max_length')
        n_req = data.get('n_requested')
        analysis = data.get('analysis', {})
        samples = data.get('samples', [])
        delimiter_issues = analyze_delimiter_issues(samples)

        # read config
        cfg = data.get('config', {}) or {}
        sim = (analysis.get('similarity') or {}) if isinstance(analysis, dict) else {}

        # Make safe types
        ts_dt = pd.to_datetime(ts, errors='coerce')

        # correct character-based ratios
        total_chars = sum(len(s or "") for s in samples)

        row = {
            'timestamp': ts_dt,
            'k': k,
            'seed': seed,
            'max_length': max_len,
            'n_requested': n_req,
            'unique_count': analysis.get('unique_count'),
            'total_count': analysis.get('total_count'),
            'avg_length': analysis.get('avg_length'),
            'hyphen_count': analysis.get('hyphen_count'),
            'underscore_count': analysis.get('underscore_count'),
            'ends_with_delimiter_ratio': delimiter_issues['ends_with_delimiter_ratio'],
            'short_segment_ratio': delimiter_issues['short_segment_ratio'],
            'all_start_with_seed': analysis.get('all_start_with_seed'),
            'empty_count': analysis.get('empty_count'),

            # config 
            'temperature': cfg.get('temperature'),
            'use_eos': cfg.get('use_eos'),
            'generator_type': cfg.get('generator_type'),
            'consonant_vowel_ratio': analysis.get('consonant_vowel_ratio'),
            'enable_trim_v1': cfg.get('enable_trim_v1'),
            'enable_trim_v2': cfg.get('enable_trim_v2'),
            'use_eos_continuation_search': cfg.get('use_eos_continuation_search'),
            'max_continuation_attempts': cfg.get('max_continuation_attempts'),

            # speed/size + similarity
            'generation_time_ms': analysis.get('generation_time_ms'),
            'throughput_samples_per_sec': analysis.get('throughput_samples_per_sec'),
            'levenshtein_mean': sim.get('levenshtein_mean'),
            'ngram_f1_mean': sim.get('ngram_f1_mean'),
            'data_size': analysis.get('data_size'),

            # denominator for char-based ratios
            'total_chars': total_chars,

            'sample_1': samples[0] if len(samples) > 0 else None,
            'sample_2': samples[1] if len(samples) > 1 else None,
            'sample_3': samples[2] if len(samples) > 2 else None,
            'file': str(p)
        }
        rows.append(row)
    except Exception as e:
        print(f"⚠️  Failed to parse {p}: {e}")

df = pd.DataFrame(rows).sort_values('timestamp')

if df.empty:
    print("No logs parsed. Run the generator with debug enabled first.")
else:
    # Derived ratios (minimal fixes)
    df['hyphen_ratio'] = df.apply(lambda r: (r.hyphen_count or 0) / r.total_chars if r.total_chars else 0, axis=1)
    df['unique_ratio'] = df.apply(lambda r: (r.unique_count or 0) / r.total_count if r.total_count else 0, axis=1)
    df['underscore_ratio'] = df.apply(lambda r: (r.underscore_count or 0) / r.total_chars if r.total_chars else 0, axis=1)

    # Fill NaN values for continuation fields
    df['use_eos_continuation_search'] = df['use_eos_continuation_search'].fillna(False)
    df['max_continuation_attempts'] = df['max_continuation_attempts'].fillna(0)

    display(df.tail(10))

Found 358 log file(s)


C:\Users\Vengeance\AppData\Local\Temp\ipykernel_21156\1123376620.py:86: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['use_eos_continuation_search'] = df['use_eos_continuation_search'].fillna(False)


,timestamp,k,seed,max_length,n_requested,unique_count,total_count,avg_length,hyphen_count,underscore_count,ends_with_delimiter_ratio,short_segment_ratio,all_start_with_seed,empty_count,temperature,use_eos,generator_type,consonant_vowel_ratio,enable_trim_v1,enable_trim_v2,use_eos_continuation_search,max_continuation_attempts,generation_time_ms,throughput_samples_per_sec,levenshtein_mean,ngram_f1_mean,data_size,total_chars,sample_1,sample_2,sample_3,file,hyphen_ratio,unique_ratio,underscore_ratio
348,2025-10-13 19:39:03.400289,4,provider,25,8,8,8,22.250,8,0,0.000,0.000,True,0,0.4,True,experimental,1.819,False,True,True,7.0,14767.50,None,14.250,0.461,1517961,178,provider-service-componen,provider-base-contrib,provider-server-simple,logs\2025-10-12\generator_debug_20251013_19390...,0.044944,1.0,0.0
349,2025-10-13 19:39:19.995288,4,connector,25,8,8,8,21.625,8,0,0.000,0.000,True,0,0.4,True,experimental,1.788,False,True,True,7.0,14920.50,None,12.625,0.528,1517961,173,connector-react-native,connectorybook-plugin,connector-config-base,logs\2025-10-12\generator_debug_20251013_19391...,0.046243,1.0,0.0
350,2025-10-13 19:39:36.240290,4,resolver,25,8,8,8,22.250,8,0,0.000,0.000,True,0,0.4,True,experimental,1.739,False,True,True,7.0,14817.50,None,14.250,0.461,1517961,178,resolversion-library,resolver-client-plugin,resolver-staticsearch,logs\2025-10-12\generator_debug_20251013_19393...,0.044944,1.0,0.0
351,2025-10-13 19:39:52.026289,4,adapter,25,8,8,8,21.375,8,0,0.000,0.000,True,0,0.4,True,experimental,1.755,False,True,True,7.0,14391.50,None,14.375,0.416,1517961,171,adapter-jsx-component,adapter-binary-string,adapter-jsdoc-plugin-inte,logs\2025-10-12\generator_debug_20251013_19395...,0.046784,1.0,0.0
352,2025-10-13 19:40:05.625288,4,mapper,25,8,8,8,22.250,8,0,0.000,0.000,True,0,0.4,True,experimental,2.048,False,True,True,7.0,12269.46,None,16.250,0.332,1517961,178,mapper-status-communit,mapper-components-api,mapperjs-config-server,logs\2025-10-12\generator_debug_20251013_19400...,0.044944,1.0,0.0
353,2025-10-13 19:40:19.031296,4,reducer,25,8,8,8,20.875,8,0,0.000,0.000,True,0,0.4,True,experimental,1.521,False,True,True,7.0,12184.50,None,13.875,0.422,1517961,167,reducers-service-button,reducer-bootstrap-compone,reducer-first-node,logs\2025-10-12\generator_debug_20251013_19401...,0.047904,1.0,0.0
354,2025-10-13 19:40:32.380789,4,filter,25,8,8,8,20.500,8,0,0.000,0.000,True,0,0.4,True,experimental,1.984,False,True,True,7.0,12045.00,None,14.500,0.360,1517961,164,filter-api-wrapper,filter-proxy-server,filter-promise-components,logs\2025-10-12\generator_debug_20251013_19403...,0.048780,1.0,0.0
355,2025-10-13 19:40:45.716788,4,transformer,25,8,8,8,21.250,8,0,0.000,0.125,True,0,0.4,True,experimental,2.239,False,True,True,7.0,12119.00,None,10.250,0.644,1517961,170,transformer-state-picker,transformer-state-transfo,transformer-stream-cli,logs\2025-10-12\generator_debug_20251013_19404...,0.047059,1.0,0.0
356,2025-10-13 19:40:58.804788,4,serializer,25,8,8,8,22.000,8,0,0.125,0.000,True,0,0.4,True,experimental,1.358,False,True,True,7.0,11840.00,None,12.000,0.574,1517961,176,serializer-data-core,serializer-plugin-app-sta,serializer-maskerred,logs\2025-10-12\generator_debug_20251013_19405...,0.045455,1.0,0.0
357,2025-10-13 19:41:11.858291,4,deserializer,25,8,8,8,22.125,8,0,0.000,0.000,True,0,0.4,True,experimental,1.290,False,True,True,7.0,11852.48,None,10.125,0.668,1517961,177,deserializer-client,deserializer-json-stream,deserializer-component,logs\2025-10-12\generator_debug_20251013_19411...,0.045198,1.0,0.0


***
# <span style="color: #f91974  ; font-weight: bold; font-size: 32px;">1.</span> <span style="color:rgb(186, 176, 115)  ; font-size: 32px;"><em>Aggregated Metrics</em></span> 

This section consolidates performance data from individual generator runs into grouped summaries by configuration parameters (k-value, temperature, EOS usage, generator type, trim settings, and continuation search settings)

**Data Segmentation:**
- **Full data**: Runs using the complete training corpus (maximum token count)
- **Subset data**: Runs using reduced training data (for comparison/testing)

***Metrics computed per configuration group:***

### Quality Metrics

**N-gram F1 score**
> **Measures**: Character-level similarity between generated names and training corpus

> **How**: Computes character n-grams (typically 3-4 character sequences) from both generated outputs and training data, then calculates the harmonic mean of precision (% of generated n-grams found in training) and recall (% of training n-grams appearing in outputs). Range: 0.0 (no overlap) to 1.0 (perfect match)

> **Example**: For generated name `parser-tools` with 3-grams: `[par, ars, rse, ser, er-, r-t, -to, too, ool, ols]`. If the training corpus contains repositories like `parser-cli`, `json-tools`, `optimizer-core`, many of these 3-grams (`par`, `ars`, `rse`, `ser`, `ool`, `ols`) will match training data patterns. High overlap (F1 > 0.75) suggests the generator heavily reuses training sequences; moderate overlap (F1 = 0.55-0.70) indicates realistic combinations of familiar character patterns; low overlap (F1 < 0.50) suggests random character sequences like `xqzprs-wkjtl` that don't resemble real repository names

> **Quality Correlation**: Moderate scores (0.55-0.70) indicate realistic naming patterns without memorization. Very high (>0.75) suggests seed word dominance; very low (<0.50) indicates nonsensical combinations

**Levenshtein distance**
> **Measures**: Inter-sample diversity within a configuration

> **How**: Calculates minimum single-character edits (insertions/deletions/substitutions) needed to transform one sample into another, then averages across all sample pairs

> **Example**: Transforming `parser-cli` → `parser-core` requires 3 edits (substitute `cli` with `core`), giving a distance of 3. Low distances (3-5) occur when samples share structure like `parser-cli`, `parser-api`, `parser-core`. High distances (>6) occur between unrelated samples like `parser-cli` vs `optimizer-react` (distance ~14)

> **Quality Correlation**: Measures output diversity, which while not directly corresponding to quality, measures an important ancilliary aspect of quality. Values of 3-5 indicate consistent structural patterns with lexical variation. Values <3 suggest near-duplicates (e.g., `parser-cli`, `parser-cla`), while values >6 indicate either appropriate diversity from heterogeneous training data or random character noise. Interpret alongside n-gram F1: high diversity + low F1 = random noise; high diversity + moderate F1 = successful varied outputs


**Average output length**
> **Measures**: Mean character count of generated repository names

> **How**: Sums character counts across all samples in a configuration group and divides by total sample count

> **Example**: With `max_length=15`, a configuration generating `[parser-tools, parser-cli, parser-core]` produces lengths `[12, 10, 11]`, averaging 11 characters (73% utilization). A better configuration generating `[optimizer-react, builder-webpack, render-engine]` with lengths `[15, 14, 13]` averages 14 characters (93% utilization). The first suggests premature termination; the second indicates natural completion near the boundary without excessive truncation

> **Quality Correlation**: When compared to `max_length`, reveals space utilization efficiency. Optimal configs approach but don't consistently hit the limit (85-95% utilization), indicating natural boundaries rather than forced truncation


**Character usage ratios (hyphens, underscores)**
> **Measures**: Proportion of delimiter characters in total output

> **How**: Counts delimiter characters across all samples, divides by total character count: `(delimiter_count / total_chars)`

> **Example**: Configuration generating `[parser-tools, builder-cli, mapper-core]` has 3 hyphens across 36 characters (ratio = 0.083 or 8.3%), exceeding optimal range. Better: `[database-sync, template-engine, validator-core]` yields 3 hyphens across 47 characters (ratio = 0.064 or 6.4%), indicating balanced naming. Poor: `[parsetools, buildercli, mappercore]` has 0 hyphens (ratio = 0.00), suggesting problematic concatenation

> **Quality Correlation**: Hyphen ratio of 0.04-0.06 signals conventional multi-component patterns (`tool-cli`). Too low (<0.03) suggests concatenation; too high (>0.07) indicates fragmentation. Underscore presence may indicate training data contamination

**Consonant/vowel ratio**
> **Measures**: Balance of consonants to vowels in generated names

> **How**: Counts consonants and vowels in each sample, calculates ratio (consonants/vowels), then averages across all samples in a configuration

> **Example**: `parser-tools` has 6 consonants (p,r,s,r,t,l,s) and 4 vowels (a,e,o,o) = ratio of 1.5. A name like `xqzprs-wkjtl` would have a much higher ratio (harder to pronounce), while `aeoiau-oiea` would be very low (unrealistic)

> **Quality Correlation**: Ratios of 1.5-2.0 indicate pronounceable, natural-sounding names. Too high (>2.5) suggests difficult pronunciation with consonant clusters; too low (<1.0) suggests unrealistic vowel-heavy patterns

### Performance Metrics

**Generation performance (time, throughput)**
> **Measures:** Computational efficiency of generation process

> **How**: Time measured in milliseconds per batch; throughput calculated as samples generated per second, accounting for batching overhead (note: throughput not implemented)


**Number of runs per configuration group**
> **Measures**: Statistical coverage of each configuration

> **How**: Counts unique seed words tested per configuration (each seed = one run)

These grouped metrics provide a foundation for systematic evaluation and ranking of parameter combinations in subsequent sections.

In [7]:
# SPLIT: FULL DATA VS SUBSET

full_size = df["data_size"].dropna().max() if df["data_size"].notna().any() else None
df["is_full_data"] = df["data_size"].eq(full_size) if full_size is not None else False
df["is_subset_data"] = df["data_size"].notna() & ~df["is_full_data"]

# counts
n_full   = int(df["is_full_data"].sum())
n_subset = int(df["is_subset_data"].sum())
n_other  = int((~df["data_size"].notna()).sum())  # rows with data_size missing/None

print(f"Full data size: {full_size} tokens")
print(f"Rows (full):   {n_full}")
print(f"Rows (subset): {n_subset}")
print(f"Rows (no data_size): {n_other}")

# subset stats
df_subset = df.loc[df["is_subset_data"]].copy()
subset_max = df_subset["data_size"].max() if not df_subset.empty else None
print(f"Subset max tokens: {subset_max}")

Full data size: 1517961 tokens
Rows (full):   358
Rows (subset): 0
Rows (no data_size): 0
Subset max tokens: None


In [10]:
# Define grouping columns for configuration analysis
group_cols = ['k', 'temperature', 'use_eos', 'generator_type', 'enable_trim_v1', 'enable_trim_v2', 'use_eos_continuation_search', 'max_continuation_attempts']

def summarize(df):
    # Filter out rows with all NaN metrics before grouping
    # (but keep rows with SOME valid data)
    metrics_cols = ['ngram_f1_mean', 'levenshtein_mean', 'avg_length', 'unique_ratio',
                    'hyphen_ratio', 'underscore_ratio', 'generation_time_ms', 'throughput_samples_per_sec']
    
    # Only exclude rows where ALL metrics are NaN or where total_count is 0
    valid_rows = df['total_count'] > 0
    df_valid = df[valid_rows].copy()
    
    out = (
        df_valid.groupby(group_cols, dropna=False)
         .agg(
            ngram_f1_mean=("ngram_f1_mean", "mean"),
            levenshtein_mean=("levenshtein_mean", "mean"),
            avg_length=("avg_length", "mean"),
            avg_max_length=("max_length", "mean"),
            unique_ratio=("unique_ratio", "mean"),
            hyphen_ratio=("hyphen_ratio", "mean"),
            underscore_ratio=("underscore_ratio", "mean"),
            consonant_vowel_ratio=("consonant_vowel_ratio", "mean"), 
            ends_with_delimiter_ratio=("ends_with_delimiter_ratio", "mean"),
            short_segment_ratio=("short_segment_ratio", "mean"),
            gen_time_ms=("generation_time_ms", "mean"),
            throughput=("throughput_samples_per_sec", "mean"),
            group_num_runs=("seed", "count"),
         )
         .reset_index()
    )
    # light rounding for readability - only apply to numeric columns
    for c in ["ngram_f1_mean","levenshtein_mean","avg_length","avg_max_length","unique_ratio",
              "hyphen_ratio","underscore_ratio","consonant_vowel_ratio","gen_time_ms","throughput"]:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors='coerce')
            out[c] = out[c].round(3)
    return out.sort_values(["k","temperature","use_eos","generator_type","enable_trim_v1","enable_trim_v2","use_eos_continuation_search","max_continuation_attempts"])

In [11]:
# GROUPED SUMMARIES

g_full   = summarize(df[df["is_full_data"]])                     # where data_size == max
g_subset = summarize(df[(~df["is_full_data"]) & df["data_size"].notna()])  # smaller-than-full

print("\n— Grouped (FULL data) —")
display(g_full)

print("\n— Grouped (SUBSET data) —")
display(g_subset)


— Grouped (FULL data) —


,k,temperature,use_eos,generator_type,enable_trim_v1,enable_trim_v2,use_eos_continuation_search,max_continuation_attempts,ngram_f1_mean,levenshtein_mean,avg_length,avg_max_length,unique_ratio,hyphen_ratio,underscore_ratio,consonant_vowel_ratio,ends_with_delimiter_ratio,short_segment_ratio,gen_time_ms,throughput,group_num_runs
0,3,0.8,True,experimental,False,False,True,3.0,0.174,14.388,17.888,20.00,1.0,0.052,0.003,1.810,0.050000,0.187500,8867.304,NaN,10
1,3,1.0,False,base,False,False,False,0.0,0.251,15.300,20.000,20.00,1.0,0.042,0.009,2.022,0.100000,0.162500,11083.600,NaN,10
2,3,1.0,False,base,False,True,False,0.0,0.453,10.859,17.109,20.00,1.0,0.037,0.007,1.880,0.000000,0.046875,10525.381,NaN,8
3,3,1.0,False,base,True,False,False,0.0,0.305,13.906,19.031,20.00,1.0,0.044,0.006,1.927,0.000000,0.046875,10704.436,NaN,8
4,3,1.0,True,experimental,False,False,True,1.0,0.373,12.125,17.625,20.00,1.0,0.051,0.005,1.998,0.062500,0.109375,9617.688,NaN,8
5,3,1.0,True,experimental,False,False,True,3.0,0.396,12.375,18.375,20.00,1.0,0.045,0.008,1.828,0.050000,0.175000,9018.056,NaN,10
6,3,1.0,True,experimental,False,False,True,5.0,0.462,11.469,18.344,20.00,1.0,0.043,0.004,1.774,0.125000,0.109375,10667.238,NaN,8
7,3,1.0,True,experimental,False,False,True,7.0,0.372,12.406,18.281,20.00,1.0,0.044,0.007,2.058,0.078125,0.093750,10511.500,NaN,8
8,3,1.0,True,experimental,False,True,True,3.0,0.298,12.475,17.075,20.00,1.0,0.047,0.007,1.962,0.012500,0.037500,10894.599,NaN,10
9,3,1.0,True,experimental,True,False,True,3.0,0.465,10.888,17.588,20.00,1.0,0.049,0.007,1.897,0.000000,0.062500,10676.150,NaN,10



— Grouped (SUBSET data) —


,k,temperature,use_eos,generator_type,enable_trim_v1,enable_trim_v2,use_eos_continuation_search,max_continuation_attempts,ngram_f1_mean,levenshtein_mean,avg_length,avg_max_length,unique_ratio,hyphen_ratio,underscore_ratio,consonant_vowel_ratio,ends_with_delimiter_ratio,short_segment_ratio,gen_time_ms,throughput,group_num_runs


***
# <span style="color: #f91974  ; font-weight: bold; font-size: 32px;">2.</span> <span style="color:rgb(186, 176, 115)  ; font-size: 32px;"><em>Quality</em></span> 

Building upon the aggregated metrics from the previous section, this section applies a multi-factor quality scoring algorithm to rank configurations and identify optimal parameter combinations.

**Ranking methodology:**
- Composite quality score derived from five weighted metrics: N-gram F1 (0.25), average length (0.25), Levenshtein distance (0.20), hyphen ratio (0.15), and temperature (0.15)
- Optimal ranges established through empirical observation of sample quality
- Configurations ranked by overall score to identify best-performing parameter sets

**Analysis workflow:**
1. Apply quality scoring function to all configuration groups
2. Rank configurations by composite score
3. Extract top-performing configurations for detailed sample inspection
4. Manual review of generated outputs to validate quantitative rankings

This combined quantitative-qualitative approach ensures that highest-ranked configurations produce both statistically favorable metrics and semantically coherent repository names.

### <span style="color:rgb(186, 176, 115); font-size: 24px;"><em>Multi-Factor Quality Scoring Algorithm</em></span>

Following the manual analysis of generated samples, this subsection implements a quantitative scoring system to systematically rank all configuration combinations.

**Scoring approach:**
- Composite score computed from five weighted factors based on empirical observations
- Each metric evaluated against optimal ranges derived from manual sample review
- Configurations ranked to identify parameter sets producing highest-quality outputs

**Key scoring factors:**
1. **N-gram F1 score** (25%): Moderate similarity (0.55-0.70) balances realism with diversity
    - Too high F1 scores may indicate that the seed word occupies a large portion of the generated word, rather than indicating strong quality in generated words
    - Very low F1 (<0.50) suggests the model generates tokens with insufficient resemblance to real repository names, producing nonsensical combinations
    - The optimal range ensures generated names feel authentic while avoiding repetitive patterns from training data

2. **Length utilization ratio** (25%): Efficient use of available space (0.85-0.95 of max_length)
    - The ratio of generated length to `max_length` indicates whether truncation occurs at appropriate boundaries
    - Very low ratios (<0.75) suggest the generator terminates prematurely, producing incomplete names that don't utilize the configured space
    - Ratios approaching 1.0 (>0.98) indicate the generator hits the hard limit frequently, suggesting forced truncation rather than natural completion
    - The optimal range (0.85-0.95) balances complete semantic units with graceful termination before the hard boundary—names like `mapper-core` (11/14 = 0.79) vs `filter-toggler` (14/15 = 0.93) demonstrate this trade-off
    - This metric is configuration-aware: a 10-character name is excellent at `max_length=12` but problematic at `max_length=20`
    
3. **Levenshtein distance** (20%): Low diversity (3-5) indicates consistent structural patterns
    - **Lower is better**: High inter-sample diversity (Levenshtein > 6) suggests random character-level variation rather than consistent naming conventions
    - The 3-5 character range indicates samples share common structural elements (seed preservation, delimiter usage, suffix patterns) while maintaining lexical variety
    - Configurations with Levenshtein < 3 risk generating nearly identical outputs, reducing utility for users seeking multiple distinct suggestions

4. **Hyphen ratio** (15%): Moderate usage (0.04-0.06) signals readable structure
    - The ratio of hyphen characters to total character count serves as a proxy for conventional repository naming patterns (`{seed}-{technology}` format)
    - Ratios below 0.03 suggest the model generates concatenated words without delimiters, reducing readability (e.g., `optimizerspec`)
    - Ratios above 0.07 indicate excessive hyphenation that fragments names into overly granular components

5. **Temperature penalty** (15%): Lower values (≤1.0) correlate with semantic coherence
    - Manual review revealed temperature > 1.4 introduces non-standard character sequences and incomplete morphemes
    - Temperature ≤ 1.0 produces conventional suffixes (`-react`, `-tools`, `-core`) while maintaining deterministic behavior
    - Higher temperatures sacrifice semantic validity for lexical diversity—a trade-off misaligned with repository naming requirements

The resulting rankings provide data-driven guidance for selecting optimal generator configurations.

In [12]:
# QUALITY RANKING 

def score_config(row):
    """Simple weighted scoring: lower Levenshtein + moderate F1 + good hyphen ratio"""
    score = 0
    
    # F1: prefer 0.55-0.70 (balanced)
    f1 = row.get('ngram_f1_mean', 0)
    if 0.55 <= f1 <= 0.70:
        score += 25
    elif 0.50 <= f1 < 0.55:
        score += 20
    elif 0.70 < f1 <= 0.72:
        score += 18  # Slight de-emphasis for high F1
    elif 0.50 <= f1 <= 0.75:
        score += 15
    
    # Length ratio: prefer 85-95% of max_length
    length_ratio = row.get('avg_length', 0) / row.get('avg_max_length', 1)
    if 0.85 <= length_ratio <= 0.95:
        score += 25
    elif 0.80 <= length_ratio < 0.85:
        score += 20
    elif 0.95 < length_ratio < 0.98:
        score += 15  # Still okay but approaching cap
    elif length_ratio < 0.65:
        score -= 25  # Very short, severe premature termination
    elif length_ratio < 0.75:
        score -= 15  # Too short, premature termination
    
    # Levenshtein: prefer 3-5 (consistent patterns)
    lev = row.get('levenshtein_mean', 10)
    if 4.8 <= lev <= 5.6:
        score += 20
        if 5.0 <= lev <= 5.4:  # Extra bonus for peak range
            score += 3
    elif 4.0 <= lev < 4.8 or 5.6 < lev <= 6.5:
        score += 15
    
    # Hyphen ratio: prefer 0.04-0.06
    hyphen = row.get('hyphen_ratio', 0)
    if 0.045 <= hyphen <= 0.060:
        score += 15
    elif 0.040 <= hyphen < 0.045 or 0.060 < hyphen <= 0.065:
        score += 12
    elif hyphen < 0.040:
        score += 8 # Mild penalty for under-delimited
    
    # removed temperature preference 
    # Temperature: lower is better
    #temp = row.get('temperature', 2.0)
    #if temp <= 1.0:
    #    score += 15
    #elif temp <= 1.4:
    #    score += 10

    # K-value: higher context window produces better coherence
    k = row.get('k', 0)
    if k >= 5:
        score += 25
    elif k == 4:
        score += 20
    elif k == 3:
        score += 10

    # Consonant/vowel ratio: prefer 1.5-2.0 (pronounceable)
    cv_ratio = row.get('consonant_vowel_ratio', 0)
    if cv_ratio > 0:  # Only score if ratio exists
        if 1.5 <= cv_ratio <= 2.0:
            score += 10
        elif 1.2 <= cv_ratio <= 2.3:
            score += 7
    
    # Underscore penalty: excessive underscores are problematic
    underscore = row.get('underscore_ratio', 0)
    if underscore > 0.02:  # More than 2% underscores
        score -= 5 # But keep penalty mild
    
    # Delimiter quality penalties
    ends_delim = row.get('ends_with_delimiter_ratio', 0)
    if ends_delim > 0.3:  # More than 30% end with delimiter
        score -= 8
    elif ends_delim > 0.1:
        score -= 5

    short_seg = row.get('short_segment_ratio', 0)
    if short_seg > 0.3:  # More than 40% have short segments
        score -= 8
    elif short_seg > 0.1:
        score -= 5
    
    return score

# Apply scoring to ALL configurations
g_full['quality_score'] = g_full.apply(score_config, axis=1)
g_ranked = g_full.sort_values('quality_score', ascending=False).reset_index(drop=True)

# Show ALL ranked configurations with quality_score first, no index
print(f"\n— All {len(g_ranked)} Configurations (Ranked by Quality Score) —")
display(g_ranked[['quality_score', 'k', 'temperature', 'use_eos', 'generator_type',
                   'enable_trim_v1', 'enable_trim_v2', 'use_eos_continuation_search', 
                   'max_continuation_attempts', 'ngram_f1_mean', 'levenshtein_mean', 
                   'hyphen_ratio', 'underscore_ratio', 'consonant_vowel_ratio', 
                   'avg_length', 'group_num_runs']])

# Show samples from top 10 with full configuration details
print("\n— Sample Outputs from Top 10 Configurations —")
for idx, config in g_ranked.head(10).iterrows():
    # Find matching rows in original data - FIXED TO INCLUDE ALL FILTERS
    mask = (
        (df["is_full_data"]) &
        (df["k"] == config["k"]) &
        (df["temperature"] == config["temperature"]) &
        (df["generator_type"] == config["generator_type"]) &
        (df["use_eos"] == config["use_eos"]) &
        (df["enable_trim_v1"] == config["enable_trim_v1"]) &
        (df["enable_trim_v2"] == config["enable_trim_v2"]) &
        (df["use_eos_continuation_search"] == config["use_eos_continuation_search"]) &
        (df["max_continuation_attempts"] == config["max_continuation_attempts"])
    )
    
    matching = df[mask]
    if matching.empty:
        continue
    
    # Collect samples with max_length
    samples = []
    for _, row in matching.iterrows():
        max_len = row.get('max_length', '?')
        for col in ['sample_1', 'sample_2', 'sample_3']:
            if pd.notna(row[col]):
                samples.append((f"{row['seed']} → {row[col]}", max_len))
        if len(samples) >= 8:
            break
    
    # Build configuration display
    use_eos = "✓" if config.get('use_eos', False) else "✗"
    trim_v1 = "✓" if config.get('enable_trim_v1', False) else "✗"
    trim_v2 = "✓" if config.get('enable_trim_v2', False) else "✗"
    use_continuation = "✓" if config.get('use_eos_continuation_search', False) else "✗"
    max_attempts = config.get('max_continuation_attempts', 0)
    
    print(f"\n{'='*70}")
    print(f"Configuration #{idx+1} | Quality Score: {config['quality_score']}")
    print(f"  K={config['k']} | temp={config['temperature']} | type={config['generator_type']}")
    print(f"  EOS: {use_eos} | Trim_v1: {trim_v1} | Trim_v2: {trim_v2}")
    print(f"  Continuation: {use_continuation} | Max attempts: {max_attempts}")
    print(f"{'='*70}")
    for i, (sample, max_len) in enumerate(samples[:8], 1):
        print(f"  {i}. {sample:<35} (max_len={max_len})")


— All 39 Configurations (Ranked by Quality Score) —


,quality_score,k,temperature,use_eos,generator_type,enable_trim_v1,enable_trim_v2,use_eos_continuation_search,max_continuation_attempts,ngram_f1_mean,levenshtein_mean,hyphen_ratio,underscore_ratio,consonant_vowel_ratio,avg_length,group_num_runs
0,95,4,0.6,True,experimental,False,True,True,7.0,0.553,12.138,0.045,0.002,1.802,21.638,10
1,90,4,0.8,True,experimental,False,False,True,3.0,0.635,8.688,0.049,0.004,1.850,18.188,10
2,90,4,1.0,True,experimental,False,False,True,3.0,0.554,10.062,0.045,0.008,1.724,18.462,10
3,88,4,1.0,True,experimental,False,True,True,3.0,0.574,9.175,0.037,0.005,1.948,17.275,10
4,88,4,1.2,True,experimental,True,False,True,3.0,0.570,9.516,0.036,0.004,1.974,17.766,8
5,87,4,1.0,True,experimental,True,False,True,3.0,0.539,9.925,0.042,0.004,1.999,17.725,10
6,87,4,0.8,True,experimental,True,False,True,3.0,0.510,10.531,0.053,0.003,2.027,17.906,8
7,87,5,1.2,True,experimental,False,False,True,3.0,0.512,10.462,0.043,0.007,1.953,17.862,10
8,87,4,1.2,True,experimental,False,False,True,3.0,0.567,9.800,0.040,0.007,1.823,18.100,10
9,87,4,1.0,False,base,False,True,False,0.0,0.522,9.781,0.043,0.006,1.762,17.281,8



— Sample Outputs from Top 10 Configurations —

Configuration #1 | Quality Score: 95
  K=4 | temp=0.6 | type=experimental
  EOS: ✓ | Trim_v1: ✗ | Trim_v2: ✓
  Continuation: ✓ | Max attempts: 7.0
  1. middleware → middlewaresdisplay     (max_len=25)
  2. middleware → middleware-asset_hashclie (max_len=25)
  3. middleware → middleware-parser-player (max_len=25)
  4. controller → controllery-analyzer   (max_len=25)
  5. controller → controllery-to-json    (max_len=25)
  6. controller → controllery-vue-graphiql (max_len=25)
  7. dispatcher → dispatcher-sync-itemap (max_len=25)
  8. dispatcher → dispatcher-client-form (max_len=25)

Configuration #2 | Quality Score: 90
  K=4 | temp=0.8 | type=experimental
  EOS: ✓ | Trim_v1: ✗ | Trim_v2: ✗
  Continuation: ✓ | Max attempts: 3.0
  1. middleware → middlewareutils        (max_len=20)
  2. middleware → middleware-app         (max_len=20)
  3. middleware → middlewareact-react-   (max_len=20)
  4. controller → controllersonvchephe   (max_len=20)
  

### <span style="color:rgb(186, 176, 115); font-size: 24px;"><em>All Configurations Manual Inspection</em></span>


The multi-factor quality scoring algorithm captures some aspects of output quality, but the automated metrics still fail to be fully reliable for assessing general quality of generated repository names.

Therefore, the following section presents all configurations with complete sample outputs for manual inspection.   
Through this manual evaluation, we identified optimal parameter settings for both base and experimental generators that produce high-quality repository names—these findings informed the recommended configurations documented in the project README.

Those conclusions can be found repeated here after the code section.

In [13]:
# PRINT ALL CONFIGURATIONS WITH ALL SAMPLES

print(f"\n{'='*80}")
print(f"ALL CONFIGURATIONS - COMPLETE SAMPLE OUTPUT")
print(f"{'='*80}\n")

# Sort by K (descending), then quality_score (descending), then temperature
g_sorted = g_full.sort_values(['k', 'quality_score', 'temperature'], 
                               ascending=[False, False, True]).reset_index(drop=True)

for idx, config in g_sorted.iterrows():
    # Find matching rows in original data
    mask = (
        (df["is_full_data"]) &
        (df["k"] == config["k"]) &
        (df["temperature"] == config["temperature"]) &
        (df["generator_type"] == config["generator_type"]) &
        (df["use_eos"] == config["use_eos"]) &
        (df["enable_trim_v1"] == config["enable_trim_v1"]) &
        (df["enable_trim_v2"] == config["enable_trim_v2"]) &
        (df["use_eos_continuation_search"] == config["use_eos_continuation_search"]) &
        (df["max_continuation_attempts"] == config["max_continuation_attempts"])
    )
    
    matching = df[mask]
    if matching.empty:
        continue
    
    # Collect ALL samples grouped by seed
    samples_by_seed = {}
    for _, row in matching.iterrows():
        seed = row['seed']
        max_len = row.get('max_length', '?')
        for col in ['sample_1', 'sample_2', 'sample_3']:
            if pd.notna(row[col]) and row[col]:
                if seed not in samples_by_seed:
                    samples_by_seed[seed] = {'max_len': max_len, 'outputs': []}
                samples_by_seed[seed]['outputs'].append(row[col])
    
    # Build configuration display
    use_eos = "✓" if config.get('use_eos', False) else "✗"
    trim_v1 = "✓" if config.get('enable_trim_v1', False) else "✗"
    trim_v2 = "✓" if config.get('enable_trim_v2', False) else "✗"
    use_continuation = "✓" if config.get('use_eos_continuation_search', False) else "✗"
    max_attempts = config.get('max_continuation_attempts', 0)
    
    print(f"\n{'='*80}")
    print(f"Config #{idx+1} | K={int(config['k'])} | Quality Score: {config['quality_score']}")
    print(f"  temp={config['temperature']} | type={config['generator_type']}")
    print(f"  EOS: {use_eos} | Trim_v1: {trim_v1} | Trim_v2: {trim_v2}")
    print(f"  Continuation: {use_continuation} | Max attempts: {max_attempts}")
    print(f"  Runs: {config['group_num_runs']} | F1: {config.get('ngram_f1_mean', 'N/A'):.3f} | Lev: {config.get('levenshtein_mean', 'N/A'):.1f}")
    print(f"{'='*80}")
    
    # Print all samples grouped by seed
    for seed in sorted(samples_by_seed.keys()):
        info = samples_by_seed[seed]
        print(f"\nSeed: '{seed}' (max_length={info['max_len']})")
        for i, output in enumerate(info['outputs'], 1):
            print(f"  {i}. {output}")


ALL CONFIGURATIONS - COMPLETE SAMPLE OUTPUT


Config #1 | K=5 | Quality Score: 87
  temp=1.2 | type=experimental
  EOS: ✓ | Trim_v1: ✗ | Trim_v2: ✗
  Continuation: ✓ | Max attempts: 3.0
  Runs: 10 | F1: 0.512 | Lev: 10.5

Seed: 'ansible' (max_length=20)
  1. ansible-stretchitmoc
  2. ansible-namedtus-nod
  3. ansible-utils-hook

Seed: 'bitbucket' (max_length=20)
  1. bitbucket-code
  2. bitbucketdb-im-selec
  3. bitbucketio-events2

Seed: 'circleci' (max_length=20)
  1. circleci-hosts
  2. circleci-yml
  3. circleci-ng-weatherr

Seed: 'docker' (max_length=20)
  1. dockervue-chocobos
  2. dockermydlib
  3. docker-io-js-selectr

Seed: 'github' (max_length=20)
  1. github-cluster-2
  2. github-cloud-visit-s
  3. github-metrika

Seed: 'gitlab' (max_length=20)
  1. gitlab-migrator-arti
  2. gitlab_bottom_bio
  3. gitlab_cookiexfersxf

Seed: 'jenkins' (max_length=20)
  1. jenkins-cdn-node-sdl
  2. jenkins-joystickyhea
  3. jenkins_job_report

Seed: 'kubernetes' (max_length=20)
  1. kubernet

**Suositellut parametrit**

*Base Generator*

Parhaat tulokset repo-nimien generointiin ilman muita kuin perus generattorin ja trien toiminnalisuuksia hyödyntäen, saa seuraavin kutsuin/parametrein.

Trim on ainoa lisä-toiminnallisuus jota perus generaattori voi hyödyntää. Tämä leikkaa sanojen lopusta sellaiset postfixit jotka eivät vaikuta sanojen luonnollisilta lopetuksilta, esim -ba

```bash
python -m src.main --enable-trim-v2
```

UI-syötteet
```code
- Starting letters     : trigger  (esimerkiksi)
- Max length           : 20       (tarpeeksi kirjaimia jotta generointi voi kehittää järkeviä nimityksiä)
- Markov degree k      : 4        (isompi k tarkoittaa yleisesti parempi-laatuisia tuloksia, joskin sanat eivät vältämättä ole innovatiivisia, k=4 pitää hyvän tasapainon)
- Number of suggestions: default  (5, tai oman mielen mukaan)
- Training data size   : default  (käytetään kaikki harjoitusdata)
- prefix               :          (oman mielen mukaan)
- Use EOS markers      : default  (base triessä ei eos)
```

Esimerkkitulostuksia:
> trigger-ts-server111  
> trigger-widge-to-pre  
> trigger-jinjakttv  
> trigger-rails  
> trigger-nodern  

Pidempien generaatioiden kohdalla, kuten yllä suositeltu max_len = 20, myös k=5 sattaa olla jopa suositelatava vaihtoehto.
Alla esimerkkitulostuksia:
> gatsby-remote-loader
> gatsby-request  
> gatsby-response_form  

> bootstrap-feedbackwa
> bootstrapOverflow 
> bootstrap_forman

Vertailuna voi nähdä mitä harjoitusdatassa on ollut vastaavalle sanan alulle.

![readme-6](images/readme-6.png)


*Experimental Generator*

trim_v1: Safer for preserving generated content
trim_v2: Better for cleaning obvious truncations, may be aggressive

```bash
python -m src.main --temperature 0.6 --use-eos-continuation-search --max-continuation-attempts 7 --enable-trim-v2
```

UI-syötteet
```code
- Starting letters     : controller (esimerkiksi)
- Max length           : 25         (eos antaa luonnollisia päättymiä, jolloin on hyvä antaa vielä enemmän kirjaimia generoinnille)
- Markov degree k      : 4          (kuin yllä, temperature 0.6 vaikuttaa että on todennäköisempää valita yleisimmin esiintyvät seuraajat)
- Number of suggestions: default    (5, tai oman mielen mukaan)
- Training data size   : default    (käytetään kaikki harjoitusdata)
- prefix               :            (oman mielen mukaan)
- näillä asetuksilla käytetään automaattisesti eos-trie:tä
```

Esimerkkitulostuksia:
> controllery-program  
> controllers-controller   
> controllerator-collective   
> controllery-botocol-serve  
> controller-starter   


Tällä configuraatiolla, myös esimerkiksi temperature=1 (ei vaihtelua perustodennäköisyyksiin) saatiin usein varsin hyviä generointeja.
Esimerkkitulostuksia:
> mocha-brewerwork
> mocha-calibre_omf  
> mocha-verifont-varia

***
# <span style="color: #f91974  ; font-weight: bold; font-size: 32px;">3.</span> <span style="color:rgb(186, 176, 115)  ; font-size: 32px;"><em>Performance</em></span> 

This section examines computational efficiency across all configurations.

**Analysis scope:**
- Generation time measured in milliseconds per batch (seed + samples)
- Rankings identify fastest parameter combinations

In [18]:
if 'g_full' not in globals():
    raise RuntimeError("g_full not found. Run the aggregation cell first.")

# Filter configurations with valid generation time data
perf = g_full[g_full['gen_time_ms'].notna()].copy()

if perf.empty:
    print("❌ No generation time data available in any configuration")
else:
    # Sort by generation time (ascending = faster is better)
    perf_ranked = perf.sort_values('gen_time_ms', ascending=True).reset_index(drop=True)
    
    print(f"\n{'='*80}")
    print(f" ALL CONFIGURATIONS RANKED BY GENERATION SPEED")
    print(f" Total configurations with timing data: {len(perf_ranked)}")
    print(f"{'='*80}\n")
    
    # Display all ranked configurations (similar format to quality ranking)
    display(perf_ranked[['gen_time_ms', 'k', 'temperature', 'use_eos', 'generator_type', 
                        'use_eos_continuation_search', 'max_continuation_attempts', 'group_num_runs', 'avg_length']])

    # Summary statistics
    print(f"\n{'='*80}")
    print(f" GENERATION TIME STATISTICS")
    print(f"{'='*80}\n")
    print(f"  Fastest:  {perf_ranked['gen_time_ms'].min():.2f} ms")
    print(f"  Slowest:  {perf_ranked['gen_time_ms'].max():.2f} ms")
    print(f"  Median:   {perf_ranked['gen_time_ms'].median():.2f} ms")
    print(f"  Mean:     {perf_ranked['gen_time_ms'].mean():.2f} ms")
    print(f"  Std Dev:  {perf_ranked['gen_time_ms'].std():.2f} ms")


 ALL CONFIGURATIONS RANKED BY GENERATION SPEED
 Total configurations with timing data: 39



,gen_time_ms,k,temperature,use_eos,generator_type,use_eos_continuation_search,max_continuation_attempts,group_num_runs,avg_length
0,8867.304,3,0.8,True,experimental,True,3.0,10,17.888
1,8993.445,3,1.4,True,experimental,True,3.0,10,18.050
2,9007.950,3,1.2,True,experimental,True,3.0,10,18.338
3,9018.056,3,1.0,True,experimental,True,3.0,10,18.375
4,9283.502,3,1.2,True,experimental,True,7.0,4,26.594
5,9617.688,3,1.0,True,experimental,True,1.0,8,17.625
6,10511.500,3,1.0,True,experimental,True,7.0,8,18.281
7,10525.381,3,1.0,False,base,False,0.0,8,17.109
8,10667.238,3,1.0,True,experimental,True,5.0,8,18.344
9,10676.150,3,1.0,True,experimental,True,3.0,10,17.588



 GENERATION TIME STATISTICS

  Fastest:  8867.30 ms
  Slowest:  16439.25 ms
  Median:   12019.19 ms
  Mean:     12387.07 ms
  Std Dev:  2334.50 ms


**Performance Analysis Observations**

- Larger k-values create more identical names, requiring more iterations to generate unique names.
- Longer, more complex outputs (higher avg_length) potentially face fewer exact matches with training data, reducing collision-based regeneration cycles
- Continuation search (`use_eos_continuation_search`) adds overhead but may be offset by earlier stopping via EOS detection